# Lending Club Loan Approval Optimization

Objective:
Build a cost-sensitive machine learning model to optimize loan approval decisions under asymmetric risk.

Business Problem:
A lending platform must balance credit losses with approval rates. Incorrect approvals increase default losses, while overly strict approvals reduce lending volume and revenue.

Dataset summary:

Rows:2218133

Columns:151

Target:loan_status

Default rate:

In [1]:
# import dependencies
import pandas as pd

In [4]:
# import data file
df = pd.read_csv('../accepted_2007_to_2018Q4.csv', low_memory=False)

In [6]:
df.head()
df.info()
df.describe()
df['loan_status'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB


loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [7]:
df

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260696,88985880,NaN,40000.0,40000.0,40000.0,60 months,10.49,859.56,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260697,88224441,NaN,24000.0,24000.0,24000.0,60 months,14.49,564.56,C,C4,...,NaN,NaN,Cash,Y,Mar-2019,ACTIVE,Mar-2019,10000.0,44.82,1.0
2260698,88215728,NaN,14000.0,14000.0,14000.0,60 months,14.49,329.33,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260699,Total amount funded in policy code 1: 1465324575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print(df.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [31]:
df['issue_d'].value_counts()

issue_d
Mar-2016    61992
Oct-2015    48631
May-2018    46311
Oct-2018    46305
Aug-2018    46079
            ...  
Aug-2007       74
Jul-2007       63
Sep-2008       57
Sep-2007       53
Jun-2007       24
Name: count, Length: 139, dtype: int64

In [32]:
all_issue_d = set(df['issue_d'].tolist())

In [33]:
print(all_issue_d)

{'Feb-2018', 'Jul-2008', 'Jul-2009', 'Aug-2016', 'Feb-2013', 'Mar-2015', 'Oct-2012', 'Mar-2011', 'Jul-2010', 'Oct-2017', 'Jun-2016', 'Mar-2008', 'Aug-2017', 'Nov-2012', 'Nov-2011', 'Nov-2017', 'Sep-2016', 'Jan-2011', 'Jun-2015', 'Mar-2018', 'Sep-2010', 'Jan-2014', 'Oct-2010', 'Jun-2007', 'Dec-2007', 'Apr-2011', 'Jun-2013', 'Apr-2015', 'Apr-2016', 'Oct-2018', 'Aug-2008', 'Nov-2016', 'Jul-2017', 'Sep-2017', 'May-2016', 'Jul-2018', 'Oct-2013', 'Dec-2009', 'Apr-2008', 'Jun-2018', 'Oct-2015', 'Apr-2010', nan, 'Jan-2009', 'Dec-2016', 'Apr-2014', 'Nov-2010', 'Apr-2012', 'May-2015', 'May-2010', 'Dec-2008', 'Aug-2018', 'Apr-2013', 'Sep-2018', 'Mar-2014', 'Sep-2008', 'Sep-2015', 'Jul-2007', 'Feb-2017', 'Aug-2007', 'Jan-2012', 'Jul-2013', 'Jan-2013', 'Jun-2012', 'Sep-2014', 'Nov-2008', 'Nov-2015', 'Jul-2011', 'Feb-2016', 'Jun-2014', 'Dec-2010', 'Nov-2007', 'Feb-2012', 'Jan-2018', 'Dec-2014', 'Apr-2018', 'Aug-2011', 'Oct-2009', 'Mar-2013', 'Feb-2008', 'Jul-2012', 'Jun-2010', 'Nov-2018', 'Jul-2015'

In [9]:
# keep only those with issue date 2012 and after
desired_issue_d = ['Feb-2018', 'Aug-2016', 'Feb-2013', 'Mar-2015', 'Oct-2012', 'Oct-2017', 'Jun-2016', 'Aug-2017', 'Nov-2012', 'Nov-2017', 'Sep-2016', 'Jun-2015', 'Mar-2018', 'Jan-2014', 'Jun-2013', 'Apr-2015', 'Apr-2016', 'Oct-2018', 'Nov-2016', 'Jul-2017', 'Sep-2017', 'May-2016', 'Jul-2018', 'Oct-2013', 'Jun-2018', 'Oct-2015', 'Dec-2016', 'Apr-2014', 'Apr-2012', 'May-2015', 'Aug-2018', 'Apr-2013', 'Sep-2018', 'Mar-2014', 'Sep-2015', 'Feb-2017', 'Jan-2012', 'Jul-2013', 'Jan-2013', 'Jun-2012', 'Sep-2014', 'Nov-2015', 'Feb-2016', 'Jun-2014', 'Feb-2012', 'Jan-2018', 'Dec-2014', 'Apr-2018', 'Mar-2013', 'Jul-2012', 'Nov-2018', 'Jul-2015', 'Jul-2016', 'Nov-2014', 'Dec-2018', 'Aug-2013', 'Aug-2014', 'Mar-2017', 'Sep-2013', 'Feb-2014', 'Oct-2016', 'Jan-2015', 'Dec-2013', 'May-2013', 'Mar-2012', 'Sep-2012', 'Jan-2016', 'Apr-2017', 'Mar-2016', 'Feb-2015', 'Dec-2015', 'Dec-2017', 'May-2017', 'May-2012', 'Aug-2015', 'Aug-2012', 'Jun-2017', 'Oct-2014', 'Dec-2012', 'Nov-2013', 'Jan-2017', 'May-2018', 'Jul-2014', 'May-2014']

In [10]:
df_post_2011 = df[df['issue_d'].isin(desired_issue_d)]

In [11]:
df_post_2011['issue_d'].value_counts()

issue_d
Mar-2016    61992
Oct-2015    48631
May-2018    46311
Oct-2018    46305
Aug-2018    46079
            ...  
May-2012     3400
Apr-2012     3230
Mar-2012     2914
Jan-2012     2602
Feb-2012     2560
Name: count, Length: 84, dtype: int64

In [40]:
df_post_2011

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260694,89885898,NaN,24000.0,24000.0,24000.0,60 months,12.79,543.50,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260695,88977788,NaN,24000.0,24000.0,24000.0,60 months,10.49,515.74,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260696,88985880,NaN,40000.0,40000.0,40000.0,60 months,10.49,859.56,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260697,88224441,NaN,24000.0,24000.0,24000.0,60 months,14.49,564.56,C,C4,...,NaN,NaN,Cash,Y,Mar-2019,ACTIVE,Mar-2019,10000.0,44.82,1.0


In [46]:
print(df_post_2011.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [12]:
#2014 and after, removing more years as the data gets too big
post_2013=['Feb-2018', 'Aug-2016', 'Mar-2015', 'Oct-2017', 'Jun-2016', 'Aug-2017', 'Nov-2017', 'Sep-2016', 'Jun-2015', 'Mar-2018', 'Jan-2014', 'Apr-2015', 'Apr-2016', 'Oct-2018', 'Nov-2016', 'Jul-2017', 'Sep-2017', 'May-2016', 'Jul-2018', 'Jun-2018', 'Oct-2015', 'Dec-2016', 'Apr-2014', 'May-2015', 'Aug-2018', 'Sep-2018', 'Mar-2014', 'Sep-2015', 'Feb-2017', 'Sep-2014', 'Nov-2015', 'Feb-2016', 'Jun-2014', 'Jan-2018', 'Dec-2014', 'Apr-2018', 'Nov-2018', 'Jul-2015', 'Jul-2016', 'Nov-2014', 'Dec-2018', 'Aug-2014', 'Mar-2017', 'Feb-2014', 'Oct-2016', 'Jan-2015', 'Jan-2016', 'Apr-2017', 'Mar-2016', 'Feb-2015', 'Dec-2015', 'Dec-2017', 'May-2017', 'Aug-2015', 'Jun-2017', 'Oct-2014', 'Jan-2017', 'May-2018', 'Jul-2014', 'May-2014']

In [14]:
df_post_2013 = df[df['issue_d'].isin(post_2013)]
df_post_2013['issue_d'].value_counts()

issue_d
Mar-2016    61992
Oct-2015    48631
May-2018    46311
Oct-2018    46305
Aug-2018    46079
Jul-2015    45962
Dec-2015    44343
Aug-2017    43573
Jul-2018    43089
Apr-2018    42928
Nov-2017    42343
Nov-2018    41973
Jun-2018    41533
Dec-2018    40134
Sep-2017    39713
Feb-2016    39529
Jul-2017    39415
Sep-2018    39026
Oct-2014    38783
Mar-2018    38771
Dec-2017    38154
Oct-2017    38151
Jun-2017    38087
May-2017    37681
Nov-2015    37530
Mar-2017    37181
Apr-2016    36432
Jan-2018    36347
Aug-2016    36280
Dec-2016    36183
Aug-2015    35886
Apr-2015    35427
Jan-2015    35107
Jul-2016    34696
Nov-2016    34591
Jun-2016    33019
Oct-2016    32772
Feb-2018    32746
Jan-2016    32366
May-2015    31913
Jan-2017    31835
Apr-2017    29683
Jul-2014    29306
Sep-2015    28641
Jun-2015    28485
May-2016    28403
Sep-2016    28144
Feb-2017    27763
Mar-2015    25400
Nov-2014    25054
Feb-2015    23770
May-2014    19099
Apr-2014    19071
Aug-2014    18814
Jun-2014    17179
Ma

In [ ]:
df_post_2013.to_csv('accepted_2014_to_2018Q4.csv', index=False)

In [19]:
df_post_2013['loan_status'].value_counts()

loan_status
Fully Paid            884132
Current               878311
Charged Off           233221
Late (31-120 days)     21464
In Grace Period         8435
Late (16-30 days)       4349
Default                   40
Name: count, dtype: int64

In [21]:
df_complete = df_post_2013[df_post_2013['loan_status'].isin(['Fully Paid','Charged Off'])].copy()
df_complete['target'] = df_complete['loan_status'].map({'Fully Paid':0,'Charged Off':1})

In [22]:
df_complete.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1117353 entries, 0 to 2260697
Columns: 152 entries, id to target
dtypes: float64(113), int64(1), object(38)
memory usage: 1.3+ GB


In [24]:
# deleted original dataframe since the cleaning has been done on a deep copy 
# del df 

In [25]:
df_complete['target'].value_counts(normalize=True)

target
0    0.791274
1    0.208726
Name: proportion, dtype: float64

The dataset was filtered to completed loans to ensure the target reflects final repayment outcomes.

The dataset was filtered to completed loans to ensure the target reflects final repayment outcomes.

In [28]:
df_complete.shape
df_complete.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1117353 entries, 0 to 2260697
Columns: 152 entries, id to target
dtypes: float64(113), int64(1), object(38)
memory usage: 1.3+ GB


In [27]:
df_complete.describe()

,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,...,hardship_amount,hardship_length,hardship_dpd,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,settlement_amount,settlement_percentage,settlement_term,target
count,0.0,1.117353e+06,1.117353e+06,1.117353e+06,1.117353e+06,1.117353e+06,1.117353e+06,1.116979e+06,1.117353e+06,1.117353e+06,...,5680.000000,5680.0,5680.000000,3696.000000,5680.000000,5680.000000,31075.000000,31075.000000,31075.000000,1.117353e+06
mean,NaN,1.454460e+04,1.454460e+04,1.453921e+04,1.310787e+01,4.411581e+02,7.718265e+04,1.866550e+01,3.370573e-01,6.954098e+02,...,148.091783,3.0,13.961972,412.721209,11056.472324,184.842072,5088.249351,47.802612,13.743524,2.087263e-01
std,NaN,8.833887e+03,8.833887e+03,8.831102e+03,4.828897e+00,2.653280e+02,7.269464e+04,1.172118e+01,9.130609e-01,3.180277e+01,...,129.060897,0.0,9.779176,359.818878,7496.391988,196.622238,3713.011885,7.080048,7.950262,4.063986e-01
min,NaN,1.000000e+03,1.000000e+03,7.250000e+02,5.310000e+00,1.401000e+01,0.000000e+00,-1.000000e+00,0.000000e+00,6.600000e+02,...,0.640000,3.0,0.000000,1.920000,55.730000,0.010000,44.210000,0.450000,0.000000,0.000000e+00
25%,NaN,8.000000e+03,8.000000e+03,8.000000e+03,9.490000e+00,2.486900e+02,4.600000e+04,1.204000e+01,0.000000e+00,6.700000e+02,...,53.147500,3.0,5.000000,147.007500,5066.130000,39.325000,2255.620000,45.000000,8.000000,0.000000e+00
50%,NaN,1.200000e+04,1.200000e+04,1.200000e+04,1.269000e+01,3.751400e+02,6.500000e+04,1.794000e+01,0.000000e+00,6.900000e+02,...,110.190000,3.0,15.000000,304.410000,9409.485000,121.130000,4232.000000,45.000000,15.000000,0.000000e+00
75%,NaN,2.000000e+04,2.000000e+04,2.000000e+04,1.561000e+01,5.871700e+02,9.200000e+04,2.456000e+01,0.000000e+00,7.100000e+02,...,205.080000,3.0,23.000000,570.915000,15395.922500,268.130000,6971.780000,50.000000,18.000000,0.000000e+00
max,NaN,4.000000e+04,4.000000e+04,4.000000e+04,3.099000e+01,1.719830e+03,1.099920e+07,9.990000e+02,3.900000e+01,8.450000e+02,...,943.940000,3.0,37.000000,2343.150000,39542.450000,1407.860000,30000.000000,521.350000,181.000000,1.000000e+00


In [29]:
df_complete.columns.tolist()

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

## Data Cleanup

### Leakage columns

**Payments / Recoveries**

total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_amnt

**Outstanding balances AFTER loan issued**

out_prncp
out_prncp_inv

**Post-loan timeline info**

last_pymnt_d
next_pymnt_d
last_credit_pull_d

**Post-loan credit performance**

last_fico_range_high
last_fico_range_low

**Settlement / hardship (VERY strong leakage)**

hardship_flag
hardship_type
hardship_reason
hardship_status
deferral_term

hardship_amount
hardship_start_date
hardship_end_date
payment_plan_start_date
hardship_length
hardship_dpd
hardship_loan_status

orig_projected_additional_accrued_interest
hardship_payoff_balance_amount
hardship_last_payment_amount
debt_settlement_flag
debt_settlement_flag_date

settlement_status
settlement_date
settlement_amount
settlement_percentage
settlement_term

### Drop (Noise)

id
member_id
url
desc
title

emp_title   # (too messy/high cardinality for now)

policy_code   # often constant

## Dataset Summary
- Rows: 1117353
- Columns: 152
- Default rate: 20.87%
- Numeric features:
  - loan_amnt
funded_amnt
funded_amnt_inv
int_rate
installment
annual_inc
dti
fico_range_low
fico_range_high
delinq_2yrs
inq_last_6mths
open_acc
pub_rec
revol_bal
revol_util
total_acc
mths_since_last_delinq
mths_since_last_record
mths_since_last_major_derog
- Categorical features:
  - term
grade
sub_grade
emp_length
home_ownership
verification_status
purpose
zip_code
addr_state
initial_list_status
application_type
verification_status_joint
pymnt_plan
disbursement_method
- Special cases (dates that either DROP or convert to numeric later)
  - issue_d
earliest_cr_line
sec_app_earliest_cr_line

In [31]:
leakage_cols = [
    'total_pymnt','total_pymnt_inv','total_rec_prncp','total_rec_int',
    'total_rec_late_fee','recoveries','collection_recovery_fee',
    'last_pymnt_amnt','out_prncp','out_prncp_inv',
    'last_pymnt_d','next_pymnt_d','last_credit_pull_d',
    'last_fico_range_high','last_fico_range_low',

    # hardship
    'hardship_flag','hardship_type','hardship_reason','hardship_status',
    'deferral_term','hardship_amount','hardship_start_date',
    'hardship_end_date','payment_plan_start_date','hardship_length',
    'hardship_dpd','hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount','hardship_last_payment_amount',

    # settlement
    'debt_settlement_flag','debt_settlement_flag_date',
    'settlement_status','settlement_date','settlement_amount',
    'settlement_percentage','settlement_term'
]

drop_cols = [
    'id','member_id','url','desc','title','emp_title','policy_code'
]

df_complete = df_complete.drop(columns=leakage_cols + drop_cols)

In [32]:
df_complete.to_csv('accepted_2014_to_2018Q4_CLEANED.csv', index=False)